In [44]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

In [45]:
# Repro + device
seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Dataset/model/training config
dataset_root = "../../../data/moleculenet"
dataset_name = "Tox21"      # MoleculeNet task
hidden_dim = 64
num_layers = 5
dropout = 0.5
train_eps = True
use_residual = True

batch_size = 128
lr = 1e-3
weight_decay = 1e-4
epochs = 100

Device: cuda


In [ ]:
dataset = MoleculeNet(root=dataset_root, name=dataset_name)
print(dataset)
print("Num graphs:", len(dataset))
print("Node feature dim:", dataset.num_node_features)

y0 = dataset[0].y
num_tasks = int(y0.view(-1).numel())
print("Targets per graph:", num_tasks)

perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(seed))
dataset = dataset[perm]

n = len(dataset)
n_train = int(0.6 * n)
n_val = int(0.2 * n)
n_test = n - n_train - n_val

train_dataset = dataset[:n_train]
val_dataset = dataset[n_train:n_train + n_val]
test_dataset = dataset[n_train + n_val:]

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Split sizes -> train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

Tox21(7831)
Num graphs: 7831
Node feature dim: 9
Targets per graph: 12
Split sizes -> train: 6264, val: 783, test: 784


In [47]:
class GIN(nn.Module):
    def __init__(
        self,
        num_classes,
        in_channels,
        hidden_dim=64,
        num_layers=5,
        dropout=0.0,
        train_eps=True,
        use_residual=True,
    ):
        super().__init__()
        assert num_layers >= 2

        self.dropout = dropout
        self.use_residual = use_residual

        def mlp(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.ReLU(),
                nn.Linear(out_dim, out_dim),
            )

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        # First layer
        self.convs.append(GINConv(mlp(in_channels, hidden_dim), train_eps=train_eps))
        self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Hidden layers
        for _ in range(num_layers - 1):
            self.convs.append(GINConv(mlp(hidden_dim, hidden_dim), train_eps=train_eps))
            self.norms.append(nn.BatchNorm1d(hidden_dim))

        # Same JK-style concatenation, then graph-level classifier
        self.lin = nn.Linear(hidden_dim * num_layers, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        h_list = []

        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h = conv(x, edge_index)
            h = norm(h)
            h = F.relu(h)

            if self.use_residual and i > 0 and h.shape == x.shape:
                h = h + x

            h = F.dropout(h, p=self.dropout, training=self.training)
            x = h
            h_list.append(h)

        # Concatenate node embeddings from all layers
        x = torch.cat(h_list, dim=-1)

        # Graph-level pooling for MoleculeNet
        g = global_mean_pool(x, batch)

        logits = self.lin(g)
        return logits

In [48]:
model = GIN(
    num_classes=num_tasks,
    in_channels=dataset.num_node_features,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=dropout,
    train_eps=train_eps,
    use_residual=use_residual,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCEWithLogitsLoss(reduction="none")

model

GIN(
  (convs): ModuleList(
    (0): GINConv(nn=Sequential(
      (0): Linear(in_features=9, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
    (1-4): 4 x GINConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (norms): ModuleList(
    (0-4): 5 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (lin): Linear(in_features=320, out_features=12, bias=True)
)

In [49]:
def prepare_targets(y):
    y = y.float()
    mask = ~torch.isnan(y)
    y = torch.where(y < 0, (y + 1.0) / 2.0, y)
    return y, mask


def masked_bce_loss(logits, y, mask):
    valid_logits = logits[mask]
    valid_y = y[mask]
    if valid_y.numel() == 0:
        return logits.new_tensor(0.0)
    return F.binary_cross_entropy_with_logits(valid_logits, valid_y)


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        y, mask = prepare_targets(batch.y)

        valid_logits = logits[mask]
        valid_y = y[mask]
        if valid_y.numel() == 0:
            continue

        loss = F.binary_cross_entropy_with_logits(valid_logits, valid_y)
        probs = torch.sigmoid(valid_logits)
        preds = (probs >= 0.5).float()

        total_loss += loss.item() * valid_y.numel()
        total_correct += (preds == valid_y).sum().item()
        total_count += valid_y.numel()

    if total_count == 0:
        return 0.0, 0.0
    return total_loss / total_count, total_correct / total_count


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(batch)
        y, mask = prepare_targets(batch.y)

        valid_logits = logits[mask]
        valid_y = y[mask]
        if valid_y.numel() == 0:
            continue

        loss = F.binary_cross_entropy_with_logits(valid_logits, valid_y)
        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(valid_logits)
        preds = (probs >= 0.5).float()

        total_loss += loss.item() * valid_y.numel()
        total_correct += (preds == valid_y).sum().item()
        total_count += valid_y.numel()

    if total_count == 0:
        return 0.0, 0.0
    return total_loss / total_count, total_correct / total_count

In [50]:
best_val_acc = -1.0
best_state = None

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(train_loader)
    va_loss, va_acc = evaluate(val_loader)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | "
            f"val_loss={va_loss:.4f} val_acc={va_acc:.4f}"
        )

if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc = evaluate(test_loader)
print(f"Best val acc: {best_val_acc:.4f}")
print(f"Test loss   : {test_loss:.4f}")
print(f"Test acc    : {test_acc:.4f}")

Epoch 001 | train_loss=0.3006 train_acc=0.9048 | val_loss=0.2479 val_acc=0.9207
Epoch 010 | train_loss=0.2246 train_acc=0.9251 | val_loss=0.2446 val_acc=0.9207
Epoch 020 | train_loss=0.2169 train_acc=0.9277 | val_loss=0.2231 val_acc=0.9254
Epoch 030 | train_loss=0.2115 train_acc=0.9290 | val_loss=0.2432 val_acc=0.9210
Epoch 040 | train_loss=0.2092 train_acc=0.9298 | val_loss=0.2105 val_acc=0.9279
Epoch 050 | train_loss=0.2058 train_acc=0.9299 | val_loss=0.2328 val_acc=0.9244
Epoch 060 | train_loss=0.2034 train_acc=0.9311 | val_loss=0.2150 val_acc=0.9256
Epoch 070 | train_loss=0.2019 train_acc=0.9320 | val_loss=0.2160 val_acc=0.9271
Epoch 080 | train_loss=0.1988 train_acc=0.9327 | val_loss=0.2101 val_acc=0.9271
Epoch 090 | train_loss=0.1966 train_acc=0.9325 | val_loss=0.2148 val_acc=0.9262
Epoch 100 | train_loss=0.1954 train_acc=0.9333 | val_loss=0.2045 val_acc=0.9290
Best val acc: 0.9303
Test loss   : 0.1940
Test acc    : 0.9360
